In [ ]:
import requests
from bs4 import BeautifulSoup

base_url = "https://paulgraham.com"
response = requests.get(f"{base_url}/articles.html")
soup = BeautifulSoup(response.text, "html.parser")

essay_urls = sorted({
    f"{base_url}/{a['href']}"
    for a in soup.find_all("a", href=True)
    if a["href"].endswith(".html") and a["href"] != "articles.html"
})

print(f"Found {len(essay_urls)} essay URLs")
print(essay_urls[:5])

In [ ]:
# import packages
from langchain_community.document_loaders import WebBaseLoader
from dotenv import load_dotenv
load_dotenv()

# Test Web Loader Initialisation for Paul Graham Essay Corpus

loader = WebBaseLoader('https://paulgraham.com/articles.html')

# Test doc variable creation
doc = loader.load()

/home/daridq/data-science-projects/agentic-ai/rag-pipeline/rag-pipeline-venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from langchain_community.document_loaders import WebBaseLoader
from bs4 import SoupStrainer
from dotenv import load_dotenv
import os

load_dotenv()

loader = WebBaseLoader(
    web_paths=essay_urls,
    bs_kwargs={"parse_only": SoupStrainer("p")},
    requests_kwargs={"headers": {"User-Agent": os.getenv("USER_AGENT", "rag-pipeline/1.0")}},
)

documents = loader.load()
print(f"Loaded {len(documents)} documents")

In [ ]:
import statistics

word_counts = [len(doc.page_content.split()) for doc in documents]

print(f"Total essays      : {len(documents)}")
print(f"Words min/max/mean: {min(word_counts)} / {max(word_counts)} / {statistics.mean(word_counts):.0f}")
print("\n--- Sample document ---")
print("Source :", documents[0].metadata["source"])
print("Preview:", documents[0].page_content[:500])